# Week 6 - 04: Manual Similarity vs Vector Database

## Goal
In Week 5, we manually created embeddings and compared them using cosine similarity.

In Week 6, ChromaDB does the storage and retrieval part for us.

We will compare the two approaches.


In [ ]:
!pip install chromadb sentence-transformers scikit-learn


In [2]:
import chromadb
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("all-MiniLM-L6-v2")

documents = [
    "Artificial Intelligence allows computers to perform tasks that normally require human intelligence.",
    "Machine learning is a part of AI where computers learn patterns from data.",
    "Natural language processing helps computers work with human language such as text and speech.",
    "Vector databases store embeddings and make similarity search fast.",
    "Semantic search finds information based on meaning, not only exact keywords.",
    "A timetable scheduling system can use AI to assign teachers, rooms, courses, and time slots.",
    "Constraint satisfaction problems are useful when a scheduling problem has many rules.",
    "Genetic algorithms can search for good solutions by using selection, crossover, and mutation."
]

question = "How can AI help create a timetable?"


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## Part A - Manual Week 5 style
Here we create all embeddings ourselves and calculate cosine similarity ourselves.
This is useful for learning because we can see what is happening

In [ ]:
# Create embeddings for the documents.
document_embeddings = model.encode(documents, convert_to_numpy=True)

# Create an embedding for the question.
question_embedding = model.encode(question, convert_to_numpy=True)

# Compare the question vector with every document vector.
similarities = cosine_similarity(
    [question_embedding],
    document_embeddings
)[0]

# Sort document positions from highest similarity to lowest similarity.
ranked_indices = similarities.argsort()[::-1]

print("Question:", question)
print("\nManual similarity results:")

for rank, index in enumerate(ranked_indices[:3], start=1):
    print(f"\nRank {rank}")
    print("Similarity:", round(float(similarities[index]), 4))
    print("Document:", documents[index])


Question: How can AI help create a timetable?

Manual similarity results:

Rank 1
Similarity: 0.7664
Document: A timetable scheduling system can use AI to assign teachers, rooms, courses, and time slots.

Rank 2
Similarity: 0.4855
Document: Artificial Intelligence allows computers to perform tasks that normally require human intelligence.

Rank 3
Similarity: 0.4443
Document: Machine learning is a part of AI where computers learn patterns from data.


Now ChromaDB stores the vectors and performs the nearest-neighbor search.

The main difference is that we do not need to manually manage the similarity calculation every time.

In [4]:
# Create/open a local Chroma database.
client = chromadb.PersistentClient(path="./chroma_week6_compare")

# Create/open a collection.
collection = client.get_or_create_collection(name="comparison")

# Store our documents and their embeddings.
ids = [f"doc_{i}" for i in range(len(documents))]

collection.upsert(
    ids=ids,
    documents=documents,
    embeddings=document_embeddings.tolist()
)

print("Stored:", collection.count(), "documents")


Stored: 8 documents


In [5]:
# Search using the question embedding.
results = collection.query(
    query_embeddings=[question_embedding.tolist()],
    n_results=3
)

print("ChromaDB results:")

for i, document in enumerate(results["documents"][0], start=1):
    print(f"\nRank {i}")
    print("Distance:", round(float(results["distances"][0][i - 1]), 4))
    print("Document:", document)


ChromaDB results:

Rank 1
Distance: 0.4671
Document: A timetable scheduling system can use AI to assign teachers, rooms, courses, and time slots.

Rank 2
Distance: 1.0291
Document: Artificial Intelligence allows computers to perform tasks that normally require human intelligence.

Rank 3
Distance: 1.1115
Document: Machine learning is a part of AI where computers learn patterns from data.


In [6]:
# Final mini test:
# Change the question and run this cell again.

new_question = "What technique can solve problems with many scheduling rules?"

new_embedding = model.encode(new_question).tolist()

results = collection.query(
    query_embeddings=[new_embedding],
    n_results=3
)

print("Question:", new_question)

for i, document in enumerate(results["documents"][0], start=1):
    print(f"\n{i}. {document}")


Question: What technique can solve problems with many scheduling rules?

1. Constraint satisfaction problems are useful when a scheduling problem has many rules.

2. A timetable scheduling system can use AI to assign teachers, rooms, courses, and time slots.

3. Genetic algorithms can search for good solutions by using selection, crossover, and mutation.
